In [1]:
%load_ext autoreload
%autoreload 2

import sys
sys.path.insert(0, '..')
import pandas as pd
import numpy as np

from src.features import compute_hedge_ratio, compute_kalman_hedge, compute_calendar_features, compute_spread_vol
from src.labels import get_daily_vol, get_vertical_barrier, apply_pt_sl_on_t1, get_bins, get_avg_uniqueness
from src.cv import temporal_split, WalkForwardPurgedCV, cv_score

## 1. Features

Compute all features on the full dataset before splitting.

In [8]:
df = pd.read_csv("../data/full_dataset.csv", index_col="date", parse_dates=True)
df = compute_hedge_ratio(df, 'close_corn', 'close_soybean')         # OLS (504-day window)
df = compute_kalman_hedge(df, 'close_corn', 'close_soybean')        # Kalman 3-state
df = compute_calendar_features(df)                                   # month, day_of_week
df = compute_spread_vol(df, 'close_corn', 'close_soybean', 'kf_hedge_ratio')  # EWMA vol

Window:          504 trading days
Valid rows:      3148 / 3652
Hedge ratio:     -0.0992 to 1.3937
Intercept:       -5.8013 to 23.9495
Spread mean:     0.0989
Converged in 8 iteration(s)
  phi      = 0.996003  (half-life = 250 trading days)
  sigma2_s = 0.023109
  delta    = 1.0e-05  (hyperparameter — tune via CV)
  R        = 1.0e-06  (numerical floor)
Hedge ratio:      0.2550 to 0.7351
Spread level std: 1.7017
Innovation z std: 1.1113 (ideal ~ 1.0)
Level z std:      1.2242


## 2. Labels

Triple barrier labeling (AFML Ch.3) + sample uniqueness weights (Ch.4).
Restrict to dates where OLS hedge ratio is valid (after 504-day burn-in).

In [9]:
corn        = df['close_corn']
soy         = df['close_soybean']
hedge_ratio = df['hedge_ratio']

# Only label dates where the OLS hedge ratio exists
valid_dates = df.dropna(subset=['hedge_ratio']).index

vol     = get_daily_vol(corn, soy, hedge_ratio)
t1      = get_vertical_barrier(valid_dates, num_days=150, t_events=valid_dates)
touches = apply_pt_sl_on_t1(corn, soy, hedge_ratio, t1, vol, pt_sl=[2, 2])
labels  = get_bins(touches, corn, soy, hedge_ratio)
weights = get_avg_uniqueness(labels, df.index)

print(f"\nLabeled events: {len(labels)}")
print(f"Label distribution: {labels['bin'].value_counts().to_dict()}")
print(f"Mean uniqueness: {weights.mean():.4f}")


Labeled events: 2998
Label distribution: {1: 1571, -1: 1427}
Mean uniqueness: 0.1187


## 3. Assemble X, y, t1, w

Build the full feature matrix: 4 Kalman-derived + 2 calendar + 1 spread vol + 90 weather = 97 features.
Exclude raw prices, volumes, and intermediate estimation columns (hedge_ratio, intercept, spread)
which would leak information or are not meaningful as predictors.

In [10]:
# Columns to EXCLUDE from features
exclude_cols = [
    'close_corn', 'close_soybean', 'high_corn', 'high_soybean',
    'low_corn', 'low_soybean', 'open_corn', 'open_soybean',
    'volume_corn', 'volume_soybean',
    'hedge_ratio', 'intercept', 'spread',       # OLS intermediates
    'kf_hedge_ratio', 'kf_intercept',            # Kalman intermediates
]

feature_cols = [c for c in df.columns if c not in exclude_cols]

X         = df.loc[labels.index, feature_cols].copy()
y         = labels['bin'].map({-1: 0, 1: 1})   # XGBoost needs {0, 1}
t1_series = labels['t1']
w         = weights

# Sanity check for NaNs
nan_count = X.isna().sum()
if nan_count.any():
    print(f"WARNING: NaN features found:")
    print(nan_count[nan_count > 0])
else:
    print(f"No NaN features — all {X.shape[0]} rows clean")

print(f"\nFeatures: {len(feature_cols)}")
print(f"  Weather:  {len([c for c in feature_cols if 'roll30d' in c])}")
print(f"  Kalman:   {[c for c in feature_cols if c.startswith('kf_')]}")
print(f"  Other:    {[c for c in feature_cols if 'roll30d' not in c and not c.startswith('kf_')]}")
print(f"\nShape: {X.shape}")
print(f"Label dist: {y.value_counts().sort_index().to_dict()}")

No NaN features — all 2998 rows clean

Features: 97
  Weather:  90
  Kalman:   ['kf_spread', 'kf_innovation', 'kf_z_score', 'kf_level_z']
  Other:    ['month', 'day_of_week', 'spread_vol']

Shape: (2998, 97)
Label dist: {0: 1427, 1: 1571}


## 4. Dev / Holdout Split

Reserve the last 1.5 years (375 trading days) as a final holdout set.
All model development happens on the dev set only. Holdout is touched once at the end.

In [11]:
split = temporal_split(X, y, t1_series, sample_weight=w, n_holdout=375)

Dev set:      2623 obs  (2013-09-20 → 2024-02-22)
Holdout set:   375 obs  (2024-02-23 → 2025-08-21)
Cutoff date: 2024-02-23
Dev label dist:     {0: 1252, 1: 1371}
Holdout label dist: {0: 175, 1: 200}


## 5. Baseline: 6-Feature XGBoost

Kalman + calendar + spread vol only. Default hyperparameters.
Establishes the baseline to beat.

In [12]:
from xgboost import XGBClassifier

model = XGBClassifier(
    n_estimators=100,
    max_depth=3,
    learning_rate=0.1,
    eval_metric='logloss',
    random_state=42,
)

baseline_cols = ['kf_level_z', 'kf_z_score', 'kf_spread', 'spread_vol', 'month', 'day_of_week']
split_base = temporal_split(X[baseline_cols], y, t1_series, sample_weight=w, n_holdout=375)

cv_base = WalkForwardPurgedCV(n_periods=3, t1=split_base['t1_dev'])
print("\n=== Baseline (6 features) — Accuracy ===")
res_base = cv_score(model, split_base['X_dev'], split_base['y_dev'], split_base['t1_dev'],
                    sample_weight=split_base['w_dev'], cv=cv_base)

Dev set:      2623 obs  (2013-09-20 → 2024-02-22)
Holdout set:   375 obs  (2024-02-23 → 2025-08-21)
Cutoff date: 2024-02-23
Dev label dist:     {0: 1252, 1: 1371}
Holdout label dist: {0: 175, 1: 200}

=== Baseline (6 features) — Accuracy ===
Fold    Train   Test  Purged               Train Dates                Test Dates  Train Score  Test Score
---------------------------------------------------------------------------------------------------------
1         870    874       4   2013-09-20 → 2017-03-06   2017-03-13 → 2020-08-28       0.8034      0.5092
2        1746    875       2   2013-09-20 → 2020-08-26   2020-08-31 → 2024-02-22       0.7795      0.4571
---------------------------------------------------------------------------------------------------------
Mean                                                                                   0.7915      0.4831
Std                                                                                    0.0120      0.0260


## 6. Full Feature Set XGBoost (97 features)

All weather + Kalman + calendar + spread vol. Same default hyperparameters.
Expect more overfitting but potentially better test performance if weather carries signal.

In [13]:
cv_full = WalkForwardPurgedCV(n_periods=3, t1=split['t1_dev'])

print("=== Full Feature Set (97 features) — Accuracy ===")
res_full = cv_score(model, split['X_dev'], split['y_dev'], split['t1_dev'],
                    sample_weight=split['w_dev'], cv=cv_full)

print("\n=== Full Feature Set — Neg Log Loss ===")
cv_full_ll = WalkForwardPurgedCV(n_periods=3, t1=split['t1_dev'])
res_full_ll = cv_score(model, split['X_dev'], split['y_dev'], split['t1_dev'],
                       sample_weight=split['w_dev'], cv=cv_full_ll, scoring='neg_log_loss')

=== Full Feature Set (97 features) — Accuracy ===
Fold    Train   Test  Purged               Train Dates                Test Dates  Train Score  Test Score
---------------------------------------------------------------------------------------------------------
1         870    874       4   2013-09-20 → 2017-03-06   2017-03-13 → 2020-08-28       0.9195      0.5069
2        1746    875       2   2013-09-20 → 2020-08-26   2020-08-31 → 2024-02-22       0.8877      0.5349
---------------------------------------------------------------------------------------------------------
Mean                                                                                   0.9036      0.5209
Std                                                                                    0.0159      0.0140

=== Full Feature Set — Neg Log Loss ===
Fold    Train   Test  Purged               Train Dates                Test Dates  Train Score  Test Score
-------------------------------------------------------------